In [1]:
!pip install kaggle

In [2]:
import json
#username和key改为自己的kaggle的，如果不行，就可以直接用这个
token = {"username":"ykz","key":"KGAT_851a5fffd594ad4afe4b9a3d209308da"}
with open('/content/kaggle.json', 'w') as file:
  json.dump(token, file)#json.dump类似于write

In [3]:
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle config set -n path -v /content

- path is now set to: /content


In [4]:
!kaggle datasets download -d slothkong/10-monkey-species

Dataset URL: https://www.kaggle.com/datasets/slothkong/10-monkey-species
License(s): CC0-1.0
 95% 517M/547M [00:03<00:00, 122MB/s]
100% 547M/547M [00:03<00:00, 184MB/s]


In [ ]:
!ls -lh datasets/slothkong/10-monkey-species/

total 548M
-rw-r--r-- 1 root root 548M Sep 26  2019 10-monkey-species.zip


In [5]:
!ls

datasets  kaggle.json  sample_data  wangdao_train.py


In [6]:
!pwd

/content


In [7]:
!unzip -o -d /content /content/datasets/slothkong/10-monkey-species/10-monkey-species.zip

Archive:  /content/datasets/slothkong/10-monkey-species/10-monkey-species.zip
  inflating: /content/monkey_labels.txt  
  inflating: /content/training/training/n0/n0018.jpg  
  inflating: /content/training/training/n0/n0019.jpg  
  inflating: /content/training/training/n0/n0020.jpg  
  inflating: /content/training/training/n0/n0021.jpg  
  inflating: /content/training/training/n0/n0022.jpg  
  inflating: /content/training/training/n0/n0023.jpg  
  inflating: /content/training/training/n0/n0024.jpg  
  inflating: /content/training/training/n0/n0025.jpg  
  inflating: /content/training/training/n0/n0026.jpg  
  inflating: /content/training/training/n0/n0027.jpg  
  inflating: /content/training/training/n0/n0028.jpg  
  inflating: /content/training/training/n0/n0029.jpg  
  inflating: /content/training/training/n0/n0030.jpg  
  inflating: /content/training/training/n0/n0031.jpg  
  inflating: /content/training/training/n0/n0032.jpg  
  inflating: /content/training/training/n0/n0033.jpg  


In [8]:
# 导入必要的库
import matplotlib as mpl
import matplotlib.pyplot as plt
# 在Jupyter notebook中内联显示图表
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
from tqdm.auto import tqdm  # 进度条库
import torch
import torch.nn as nn
import torch.nn.functional as F

# 打印Python版本信息
print(sys.version_info)

# 打印各个库的版本信息
for module in mpl, np, pd, sklearn, torch:
    print(module.__name__, module.__version__)

# 设置设备：如果有GPU则使用GPU，否则使用CPU
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(device)


sys.version_info(major=3, minor=12, micro=12, releaselevel='final', serial=0)
matplotlib 3.10.0
numpy 2.0.2
pandas 2.2.2
sklearn 1.6.1
torch 2.9.0+cu126
cuda:0


In [9]:
!ls  /content/training/training

n0  n1	n2  n3	n4  n5	n6  n7	n8  n9


In [10]:
!ls /content/validation/

validation


# 数据预处理

In [12]:
# INSERT_YOUR_CODE
from torchvision import datasets, transforms
import os

# 设置数据目录
data_dir = '/content/'

# 定义预处理: resize到128x128, 转为tensor
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4363, 0.4328, 0.3291], std=[0.2427, 0.2382, 0.2413])
])

# 读取训练集和测试集
train_dataset = datasets.ImageFolder(root=os.path.join(data_dir, 'training/training'), transform=transform)
test_dataset = datasets.ImageFolder(root=os.path.join(data_dir, 'validation/validation'), transform=transform)

# 获取类别名（方便后续显示标签）
class_names = train_dataset.classes
class_names

['n0', 'n1', 'n2', 'n3', 'n4', 'n5', 'n6', 'n7', 'n8', 'n9']

In [13]:
len(train_dataset)

1097

In [14]:
train_dataset[0][0].shape #特征

torch.Size([3, 128, 128])

In [15]:
train_dataset[0][1] #标签

0

In [16]:
len(test_dataset)

272

In [17]:
# INSERT_YOUR_CODE

# from torch.utils.data import DataLoader
# import torch

# loader = DataLoader(train_dataset, batch_size=64, shuffle=False, num_workers=2)

# mean = torch.zeros(3)
# std = torch.zeros(3)
# n_pixels = 0

# for images, _ in loader:  # images: [B, 3, 128, 128]
#     batch_pixels = images.numel() // 3  # total pixels per channel
#     mean += images.sum(dim=[0, 2, 3])
#     std  += (images ** 2).sum(dim=[0, 2, 3])
#     n_pixels += batch_pixels

# mean /= n_pixels
# std = torch.sqrt(std / n_pixels - mean ** 2)

# print("按通道均值:", mean)
# print("按通道标准差:", std)



In [18]:
from torch.utils.data import DataLoader

# 创建训练集和验证集的DataLoader
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,  # 训练时打乱数据
    num_workers=2  # 使用多进程加载数据
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,  # 测试时不需要打乱
    num_workers=2
)

print(f"训练集DataLoader批次数: {len(train_loader)}")
print(f"测试集DataLoader批次数: {len(test_loader)}")
print(f"每个批次大小: {batch_size}")

# 查看一个批次的数据
train_iter = iter(train_loader)
batch_images, batch_labels = next(train_iter)
print(f"批次图像张量形状: {batch_images.shape}")
print(f"批次标签张量形状: {batch_labels.shape}")
print(batch_labels)

训练集DataLoader批次数: 35
测试集DataLoader批次数: 9
每个批次大小: 32
批次图像张量形状: torch.Size([32, 3, 128, 128])
批次标签张量形状: torch.Size([32])
tensor([7, 1, 0, 1, 9, 8, 6, 0, 8, 4, 9, 0, 1, 4, 1, 9, 3, 0, 2, 6, 6, 5, 1, 1,
        4, 0, 9, 4, 2, 6, 5, 7])


In [19]:
28*28

784

在PyTorch中，`DataLoader`是一个迭代器，它封装了数据的加载和预处理过程，使得在训练机器学习模型时可以方便地批量加载数据。`DataLoader`主要负责以下几个方面：

1. **批量加载数据**：`DataLoader`可以将数据集（Dataset）切分为更小的批次（batch），每次迭代提供一小批量数据，而不是单个数据点。这有助于模型学习数据中的统计依赖性，并且可以更高效地利用GPU等硬件的并行计算能力。

2. **数据打乱**：默认情况下，`DataLoader`会在每个epoch（训练周期）开始时打乱数据的顺序。这有助于模型训练时避免陷入局部最优解，并且可以提高模型的泛化能力。

3. **多线程数据加载**：`DataLoader`支持多线程（通过参数`num_workers`）来并行地加载数据，这可以显著减少训练过程中的等待时间，尤其是在处理大规模数据集时。

4. **数据预处理**：`DataLoader`可以与`transforms`结合使用，对加载的数据进行预处理，如归一化、标准化、数据增强等操作。

5. **内存管理**：`DataLoader`负责管理数据的内存使用，确保在训练过程中不会耗尽内存资源。

6. **易用性**：`DataLoader`提供了一个简单的接口，可以很容易地集成到训练循环中。

# 搭建模型

In [20]:
128//8

16

In [21]:
import torch.nn as nn

class MonkeyNet(nn.Module):
    def __init__(self, num_classes=10):
        super(MonkeyNet, self).__init__()

        # 针对10-monkeys高分辨率彩色图片(3通道，128x128)，构建更深更宽更正则化的卷积网络
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),   # (B,32,128,128)
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),  # (B,32,128,128)
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                                       # (B,32,64,64)


            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),  # (B,64,64,64)
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),  # (B,64,64,64)
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                                       # (B,64,32,32)


            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False), # (B,128,32,32)
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1, bias=False),# (B,128,32,32)
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                                       # (B,128,16,16)


            nn.Conv2d(128, 256, kernel_size=3, padding=1, bias=False), # (B,256,16,16)
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1, bias=False),# (B,256,16,16)
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2,2),                                     # (B,256,8,8)

        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*8*8, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# 实例化模型
model = MonkeyNet(num_classes=10)


In [22]:
128*16*16*256

8388608

In [23]:
# 使用随机输入对模型进行一次前向计算以验证模型结构是否正确
import torch

dummy_input = torch.randn(32, 3, 128, 128)
output = model(dummy_input) #前向传播/前向计算/正向传播
print(f"Output shape: {output.shape}")


Output shape: torch.Size([32, 10])


In [24]:
# 输出model每一层的参数量
total_params = 0  # 初始化总参数量为0
print("各层参数量统计：")  # 打印参数统计表头
for name, param in model.named_parameters():  # 遍历模型中所有需要优化的参数
    if param.requires_grad:  # 只有需要梯度更新的参数才统计
        num_params = param.numel()  # 计算当前参数的元素总数
        total_params += num_params  # 更新总参数量
        print(f"{name}: {num_params}")  # 输出当前层的参数量
print(f"模型总参数量: {total_params}")  # 输出模型总参数量


各层参数量统计：
features.0.weight: 864
features.1.weight: 32
features.1.bias: 32
features.3.weight: 9216
features.4.weight: 32
features.4.bias: 32
features.7.weight: 18432
features.8.weight: 64
features.8.bias: 64
features.10.weight: 36864
features.11.weight: 64
features.11.bias: 64
features.14.weight: 73728
features.15.weight: 128
features.15.bias: 128
features.17.weight: 147456
features.18.weight: 128
features.18.bias: 128
features.21.weight: 294912
features.22.weight: 256
features.22.bias: 256
features.24.weight: 589824
features.25.weight: 256
features.25.bias: 256
classifier.1.weight: 2097152
classifier.1.bias: 128
classifier.3.weight: 1280
classifier.3.bias: 10
模型总参数量: 3271786


In [25]:
32*3*3*64

18432

# 训练

In [26]:
import torch.nn as nn
import torch.optim as optim

# 初始化交叉熵损失函数，内部会做softmax
criterion = nn.CrossEntropyLoss()

# 初始化优化器（这里选用Adam，也可以使用SGD等）
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [27]:
import wangdao_train
# 假设train_loader和val_loader已定义，device已经设为"cuda"或"cpu"
trainer = wangdao_train.Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    eval_step=100
)

# 设定训练轮数
num_epochs = 20

# 开始训练
trainer.train(num_epochs)


Epoch [1/20]  Train Loss: 3.0325  Train Acc: 0.1996
Epoch [2/20]  Train Loss: 1.9876  Train Acc: 0.2689
[Step 100] Val Loss: 1.7717 Val Acc: 0.3015
Epoch [3/20]  Train Loss: 1.7641  Train Acc: 0.3446
Epoch [4/20]  Train Loss: 1.6248  Train Acc: 0.4057
Epoch [5/20]  Train Loss: 1.4791  Train Acc: 0.4576
[Step 200] Val Loss: 1.5302 Val Acc: 0.4338
Epoch [6/20]  Train Loss: 1.3276  Train Acc: 0.5251
Epoch [7/20]  Train Loss: 1.2209  Train Acc: 0.5688
Epoch [8/20]  Train Loss: 1.0270  Train Acc: 0.6372
[Step 300] Val Loss: 1.4815 Val Acc: 0.4853
Epoch [9/20]  Train Loss: 1.0113  Train Acc: 0.6345
Epoch [10/20]  Train Loss: 0.8601  Train Acc: 0.6855
Epoch [11/20]  Train Loss: 0.7410  Train Acc: 0.7256
[Step 400] Val Loss: 1.5329 Val Acc: 0.5588
Epoch [12/20]  Train Loss: 0.7638  Train Acc: 0.7183
Epoch [13/20]  Train Loss: 0.6371  Train Acc: 0.7694
Epoch [14/20]  Train Loss: 0.5836  Train Acc: 0.7967
[Step 500] Val Loss: 1.3568 Val Acc: 0.5882
Epoch [15/20]  Train Loss: 0.5147  Train Acc: 0

In [28]:
trainer.train(num_epochs)

Epoch [1/20]  Train Loss: 0.2209  Train Acc: 0.9344
Epoch [2/20]  Train Loss: 0.1666  Train Acc: 0.9453
[Step 100] Val Loss: 1.4579 Val Acc: 0.6287
Epoch [3/20]  Train Loss: 0.1706  Train Acc: 0.9407
Epoch [4/20]  Train Loss: 0.1017  Train Acc: 0.9690
Epoch [5/20]  Train Loss: 0.1044  Train Acc: 0.9690
[Step 200] Val Loss: 1.3938 Val Acc: 0.6801
Epoch [6/20]  Train Loss: 0.0704  Train Acc: 0.9818
Epoch [7/20]  Train Loss: 0.0831  Train Acc: 0.9818
Epoch [8/20]  Train Loss: 0.1368  Train Acc: 0.9535
[Step 300] Val Loss: 1.6310 Val Acc: 0.6801
Epoch [9/20]  Train Loss: 0.0931  Train Acc: 0.9690
Epoch [10/20]  Train Loss: 0.2685  Train Acc: 0.9152
Epoch [11/20]  Train Loss: 0.2266  Train Acc: 0.9180
[Step 400] Val Loss: 1.6233 Val Acc: 0.6691
Epoch [12/20]  Train Loss: 0.1451  Train Acc: 0.9499
Epoch [13/20]  Train Loss: 0.0647  Train Acc: 0.9827
Epoch [14/20]  Train Loss: 0.0415  Train Acc: 0.9918
[Step 500] Val Loss: 1.7775 Val Acc: 0.6875
Epoch [15/20]  Train Loss: 0.0280  Train Acc: 0